In [1]:
import pandas as pd

In [7]:
from data_loader import load_data, load_production_raw

In [8]:
df_full = load_data()
prod_raw = load_production_raw()

In [13]:
import glob
import os
import pandas as pd
fil = "C:\\Users\\MelBl\\OneDrive - datadelver.io\\Documents\\ny_wells\\"

DATA_DIR = os.path.join(os.path.dirname(fil), "src", "data")

WELL_FILES_PATTERN = os.path.join(DATA_DIR, "[0-9]*.csv")
PRODUCTION_FILE = os.path.join(DATA_DIR, "well-production.csv")

DATE_COLS = [
    "Status Date",
    "Permit Application Date",
    "Permit Issued Date",
    "Spud/Start Drilling Date",
    "Total Depth Date",
    "Well Completion Date",
    "Plugging & Abandonment Date",
    "Last Modified Date",
]

NUMERIC_COLS = [
    "Proposed Total Depth",
    "Surface Longitude",
    "Surface Latitude",
    "Bottom Hole Longitude",
    "Bottom Hole Latitude",
    "True Vertical Depth",
    "Bottom Hole Total Measured Depth",
    "Kickoff Depth",
    "Drilled Depth",
]

PRODUCTION_NUMERIC = ["OIL (Bbls)", "GAS (Mcf)", "WATER (Bbls)", "Months in Production"]


def _strip_df(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip()
    obj_cols = df.select_dtypes(include="object").columns
    df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())
    return df


def load_production_raw() -> pd.DataFrame:
    prod = pd.read_csv(PRODUCTION_FILE, dtype=str, low_memory=False)
    prod = _strip_df(prod)
    prod = prod.drop(columns=[c for c in prod.columns if c == ""], errors="ignore")
    for col in PRODUCTION_NUMERIC:
        if col in prod.columns:
            prod[col] = pd.to_numeric(prod[col], errors="coerce")
    prod["Year"] = pd.to_numeric(prod["Year"], errors="coerce")
    return prod


def load_data(prod_raw: pd.DataFrame) -> pd.DataFrame:
    files = sorted(glob.glob(WELL_FILES_PATTERN))
    frames = [pd.read_csv(f, dtype=str, low_memory=False, on_bad_lines="skip") for f in files]
    df = pd.concat(frames, ignore_index=True)

    df = _strip_df(df)

    # drop any duplicate columns introduced by malformed rows
    df = df.loc[:, ~df.columns.duplicated()]

    df.drop_duplicates(subset=["API Well Number"], keep="last", inplace=True)
    df.reset_index(drop=True, inplace=True)

    for col in DATE_COLS:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["Spud Year"] = df["Spud/Start Drilling Date"].dt.year

    df = df[
        (df["Surface Longitude"].between(-80, -72))
        & (df["Surface Latitude"].between(40, 46))
        | df["Surface Longitude"].isna()
    ]

#    prod = prod_raw.groupby("API Well Number", as_index=False).agg(
#        Total_Oil_Bbls=("OIL (Bbls)", "sum"),
#        Total_Gas_Mcf=("GAS (Mcf)", "sum"),
#        Total_Water_Bbls=("WATER (Bbls)", "sum"),
#        Production_Years=("Year", "nunique"),
#        Latest_Production_Year=("Year", "max"),
#    )
 
    df = df.merge(prod_raw, on="API Well Number", how="left")

#    prod_cols = ["Total_Oil_Bbls", "Total_Gas_Mcf", "Total_Water_Bbls",
#                 "Production_Years", "Latest_Production_Year"]
#    df[prod_cols] = df[prod_cols].fillna(0)

    return df


In [14]:
prod_raw = load_production_raw()
df=  load_data(prod_raw)

C:\Users\MelBl\AppData\Local\Temp\ipykernel_10508\141372849.py:39: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns
C:\Users\MelBl\AppData\Local\Temp\ipykernel_10508\141372849.py:39: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

In [21]:
df[df["OIL (Bbls)"].notna()]

,API Well Number,Production Information,Formation Tops,Casing and Cementing,Hole Number,Well Name_x,Company Name,Well Type_x,Well Status,Objective Formation,...,Last Modified Date,Spud Year,Well Name_y,Operator Name,Year,OIL (Bbls),GAS (Mcf),WATER (Bbls),Months in Production,Well Type_y
29,31-003-04825-00-00,Yes,Yes,No,4825,2 L. Potter,Outman Joshua J.,Oil Development,Inactive,Not Applicable,...,2025-07-08,NaN,2 L. Potter,Outman Joshua J.,2024.0,0.0,0.0,0.0,0.0,Oil Development
30,31-003-04958-00-00,Yes,Yes,Yes,4958,Slade Lot 29 F-23,Sandstone Oil & Gas Development LLC,Oil Development,Inactive,Richburg,...,2025-07-08,1966.0,Slade Lot 29 F-23,Sandstone Oil & Gas Development LLC,2024.0,0.0,0.0,0.0,0.0,Oil Development
31,31-003-04959-00-00,Yes,Yes,Yes,4959,Slade Lot 29 B-11,Sandstone Oil & Gas Development LLC,Enhanced Oil Recovery - Injection,Inactive,Richburg,...,2019-12-13,1966.0,Slade Lot 29 B-11,Sandstone Oil & Gas Development LLC,2024.0,0.0,0.0,0.0,0.0,Enhanced Oil Recovery - Injection
32,31-003-04960-00-00,Yes,Yes,Yes,4960,Slade Lot 29 G-2,Sandstone Oil & Gas Development LLC,Enhanced Oil Recovery - Injection,Inactive,Richburg,...,2019-12-13,1966.0,Slade Lot 29 G-2,Sandstone Oil & Gas Development LLC,2024.0,0.0,0.0,0.0,0.0,Enhanced Oil Recovery - Injection
33,31-003-04961-00-00,Yes,Yes,Yes,4961,Slade Lot 29 H-3,Sandstone Oil & Gas Development LLC,Oil Development,Inactive,Richburg,...,2025-07-08,1966.0,Slade Lot 29 H-3,Sandstone Oil & Gas Development LLC,2024.0,0.0,0.0,0.0,0.0,Oil Development
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15999,31-101-30015-00-00,Yes,Yes,Yes,30015,Black bear 1,Wildcat Oil Producer LLC,Oil Development,Active,Fulmer Valley,...,2025-12-05,2022.0,Black bear 1,Wildcat Oil Producer LLC,2024.0,154.0,0.0,30.0,12.0,Oil Development
16018,31-107-26447-00-00,Yes,Yes,Yes,26447,Tubbs 2,Epsilon Energy USA Inc.,Gas Wildcat,Inactive,Oriskany,...,2025-12-05,2010.0,Tubbs 2,Epsilon Energy USA Inc.,2024.0,0.0,0.0,0.0,0.0,Gas Wildcat
16030,31-121-14804-00-01,Yes,Yes,No,14804,Ludwig 1A,PPP Future Development Inc.,Gas Wildcat,Active,Marcellus,...,2025-12-05,2010.0,Ludwig 1A,PPP Future Development Inc.,2024.0,0.0,14.0,0.0,6.0,Gas Wildcat
16031,31-121-16330-00-01,Yes,Yes,Yes,16330,Reisdorf 1A,PPP Future Development Inc.,Gas Wildcat,Active,Marcellus,...,2025-12-05,2010.0,Reisdorf 1A,PPP Future Development Inc.,2024.0,0.0,3989.0,0.0,10.0,Gas Wildcat
